# OSM 权力到形态:看权力如何改写城市天际线

逐格执行的练习,**不用写程式**,按顺序跑即可。程式码都在 engine 资料夹,平时不用打开。

做的事:取一批真实城市建筑(每栋有占地 footprint 与高度)→ 给每栋贴角色(政府 state、开发商 developer、居民 resident、在地 informal)→ 切换权力情景,看天际线怎么变。

一条规则:**占地与角色标签固定不变,只有高度会改**。样本:新加坡大巴窑,1163 栋,离线纯 Python。

## 怎么执行

- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。
- 跟着说明,在标 ✏️ 的地方改设定,再重跑那一步。

In [ ]:
# 讓練習本找到 engine 裡的程式碼(這格不用改,直接執行)
import sys, os
for _p in (".", "engine", os.path.join("engine", "steps")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

## 你可以改的两个档(用记事本就能开)

- `stakeholder_lookup.yaml` — 哪种建筑算哪个角色。改完重跑 **Step 2**。
- `power_scenarios.yaml` — 每个角色的高度政策(mult、cap_m、_mode)。改完重跑 **Step 3、4**。

诚实边界:高度多为估算;用途不明标 unknown 不猜;这是可编辑对照表,非产权认定。

## 准备

第一格装套件(已装秒过),第二格汇入工具并读设定,印出建筑数即就绪。

In [ ]:
import deps; deps.ensure()   # 缺套件就裝(已裝秒過);換地方需要 osmnx

In [ ]:
import importlib, config; importlib.reload(config)   # 改完 config.py 回來重跑這格
import common, plots, geo                            # common=算 / plots=畫 / geo=座標
import matplotlib; matplotlib.use("module://matplotlib_inline.backend_inline")
%matplotlib inline
from IPython.display import Image, display

df = common.current_buildings()                      # 載入這塊地的建築(還沒貼角色)
print("PLACE =", config.PLACE, "| UTM =", config.UTM, "| 建築數 =", len(df))
print("資料來源:", "bundled 大巴窯(離線)" if config.BUNDLED else config.PLACE)
print(common.honest_note())

## 换地方(可选)

1. 地图右键复制目标座标(纬度、经度)。
2. 开 `config.py`,填 `LAT`、`LON`、`RADIUS_M`,存档。
3. 重跑 Run All,整条流程切到新地点(BBOX/UTM 自动推导,需联网 + osmnx)。

下面这格先预览会得到的范围。

In [ ]:
# 換地方前先預覽範圍(把 lat/lon/半徑換成你的目標,確認 OK 再去 config.py 填)
geo.preview(lat=35.6780, lon=139.7782, radius_m=1300, place="Tokyo")

## Step 0(可选):卫星底图

先看研究范围真实的样子,再看后面的抽象图。需要 contextily 和网路,没有就跳过。

In [ ]:
# Step 0(可選):衛星底圖 + figure-ground,存三張 PNG 並顯示。需 contextily/pyproj + 網路。
try:
    import step0_satellite as s0
    for p in s0.run(df=common.assign_all(df)):
        print(p.name); display(Image(str(p)))
except Exception as e:
    print("Step 0 跳過(沒網路或缺套件,不影響後面):", type(e).__name__, e)

## Step 1:读建筑占地

看这批建筑的占地与高度分布。这步**还没上色**,全灰,先看清原料。高度多为楼层估算,用途标签很稀疏——所以后面用对照表而非推断。

In [ ]:
n = len(df)
print("建築數:", n, "| 總 footprint 面積 %.0f m²" % df.area_m2.sum())
print("高度:平均 %.1f m / 最矮 %.1f m / 最高 %.1f m" % (df.height_m.mean(), df.height_m.min(), df.height_m.max()))
print("height_source(測量 vs 估計):", df.height_source.value_counts().to_dict())
plots.data_overview(df)        # 左:灰 footprints;右:高度分佈

## Step 2:贴角色标签

按 `stakeholder_lookup.yaml` 查表给每栋一个角色,第一个命中为准,用途不明标 unknown。只加标签,不动几何与高度。

In [ ]:
df = common.assign_all(df)     # 加 'stakeholder' 欄(改 stakeholder_lookup.yaml 可變這步)
print("各角色棟數:", df["stakeholder"].value_counts().to_dict())
plots.power_map(df)            # 左:角色著色;右:棟數/面積佔比

## Step 3:权力情景

每个情景给各角色一个高度权重 mult、可选上限 cap_m,与模式 _mode:
- **conserve**:总建筑量不变,只在角色间重分配。
- **grow**:以现有高度为下限往上加,总量上升。

后面所有形态差异都来自这张表。改 `power_scenarios.yaml` 重跑这格。

In [ ]:
scen = common.load_scenarios()
print("可用情景:", list(scen.keys()))
plots.policy_heatmap(scen)     # 乘數熱圖:紅=拔高 / 藍=壓低 / 白=照舊

## Step 4:套情景,只调高度

同一批建筑、同样占地与标签,分别套几个情景只改高度。因为占地与标签锁定,天际线差异**完全来自权力配置**。

想看明显变化用 grow 情景(如 `max_change`);conserve 总量守恒,只小幅起伏。

In [ ]:
SCEN_NAMES = ["current", "community_led", "developer_renewal", "max_change"]   # ✏️ 可改要比的情景
heights = {name: common.scenario_heights(df, scen[name]) for name in SCEN_NAMES}
plots.skyline_panels(df, heights, names=SCEN_NAMES)   # 四連幅:只調高度的新形態
plots.metrics(df, heights, names=SCEN_NAMES)          # 指標:總GFA / 平均高 / 最高 / 各角色

## Step 5:汇出 3D 量体

把占地按高度挤成 3D,萤幕预览并汇出 OBJ 到 `engine/out`。想要可拖动旋转的 3D / plotly,看「进阶-变成3D」那本。

In [ ]:
sub = df.sort_values("area_m2", ascending=False).head(80).copy()   # 取佔地最大 80 棟示範
plots.city_3d(sub)                                                 # 螢幕預覽 3D
obj, nv, nf = common.extrude_obj(sub, height_col="height_m")       # 真實 footprint 擠成量體
(common.OUT / "city_current_demo.obj").write_text(obj, encoding="utf-8")
print("→ out/city_current_demo.obj  頂點%d 面%d" % (nv, nf))
print("完整匯出(全部樓/全部情景):終端跑 python engine/steps/step5_export_rhino.py")

## 动手练习

改一个设定,重跑对应步骤,看天际线变化:
- 改角色归属 → `stakeholder_lookup.yaml` → 重跑 Step 2。
- 改高度政策 → `power_scenarios.yaml`(mult / cap_m / 新情景)→ 重跑 Step 3、4。

In [ ]:
print("管線走完。兩張可編輯 YAML:")
print("  1) stakeholder_lookup.yaml — 誰算誰的(改完重跑 Step 2)")
print("  2) power_scenarios.yaml    — 權力如何改寫高度(改完重跑 Step 3、4)")
print("固定不變:footprint + 角色標籤;只有高度會動。")
print(common.honest_note())